In [8]:
pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install imageio

Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
import imageio
import copy
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
# Image Loader
def load_image(path, size):
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor()
    ])
    image = Image.open(path).convert("RGB")
    image = transform(image).unsqueeze(0)
    return image.to(device)

def tensor_to_image(tensor):
    image = tensor.clone().detach().cpu().squeeze(0)
    return transforms.ToPILImage()(image)


In [12]:
# Content Loss
class ContentLoss(nn.Module):
    def __init__(self, target):
        super().__init__()
        self.target = target.detach()

    def forward(self, x):
        self.loss = nn.functional.mse_loss(x, self.target)
        return x   

In [13]:
# Style Loss
class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super().__init__()
        self.target = self.gram_matrix(target_feature).detach()

    def gram_matrix(self, x):
        b, c, h, w = x.size()
        features = x.view(c, h * w)
        G = torch.mm(features, features.t())
        return G / (c * h * w)

    def forward(self, x):
        G = self.gram_matrix(x)
        self.loss = nn.functional.mse_loss(G, self.target)
        return x   

In [14]:
# Normalization
class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        self.mean = torch.tensor(mean).view(-1,1,1).to(device)
        self.std = torch.tensor(std).view(-1,1,1).to(device)

    def forward(self, img):
        return (img - self.mean) / self.std


In [15]:
# Build Model with Loss Layers
def get_model_and_losses(cnn, content_img, style_img):

    normalization_mean = [0.485, 0.456, 0.406]
    normalization_std = [0.229, 0.224, 0.225]

    cnn = copy.deepcopy(cnn)
    normalization = Normalization(normalization_mean, normalization_std)

    content_layers = ['conv_4']
    style_layers = ['conv_1','conv_2','conv_3','conv_4','conv_5']

    model = nn.Sequential(normalization)
    content_losses = []
    style_losses = []

    i = 0
    for layer in cnn.children():

        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f'conv_{i}'
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f'pool_{i}'
        else:
            continue

        model.add_module(name, layer)

        if name in content_layers:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module(f"content_loss_{i}", content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module(f"style_loss_{i}", style_loss)
            style_losses.append(style_loss)

    return model, style_losses, content_losses


In [16]:
def run_style_transfer(content_path, style_path,
                       size=512,
                       steps=300,
                       style_weight=1e6,
                       content_weight=1):

    # Load images
    content_img = load_image(content_path, size)
    style_img = load_image(style_path, size)

    input_img = content_img.clone().requires_grad_(True)


    cnn = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()

    # Build model with losses
    model, style_losses, content_losses = get_model_and_losses(
        cnn, content_img, style_img)

    model.eval()
    optimizer = optim.LBFGS([input_img])

    frames = []
    run = 0

    while run <= steps:

        def closure():
            nonlocal run

            with torch.no_grad():
                input_img.clamp_(0, 1)

            optimizer.zero_grad()

            model(input_img)

            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss

            for cl in content_losses:
                content_score += cl.loss

            loss = style_weight * style_score + content_weight * content_score
            loss.backward()

            run += 1

            if run % 50 == 0:
                print(f"Step {run} | Style: {style_score.item():.4f} | Content: {content_score.item():.4f}")
                frames.append(tensor_to_image(input_img.detach()))

            return loss

        optimizer.step(closure)

    with torch.no_grad():
        input_img.clamp_(0, 1)

    output = tensor_to_image(input_img.detach())

    # results
    import os
    os.makedirs("results", exist_ok=True)

    output.save("results/output.jpg")

    if len(frames) > 0:
        import imageio
        imageio.mimsave("results/progress.gif", frames, duration=0.5)

    # Display results
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(tensor_to_image(content_img))
    plt.title("Content")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(tensor_to_image(style_img))
    plt.title("Style")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(output)
    plt.title("Output")
    plt.axis("off")

    plt.show()

In [17]:
def tensor_to_image(tensor):
    image = tensor.clone().detach().cpu().squeeze(0)
    return transforms.ToPILImage()(image)

In [ ]:
class ContentLoss(nn.Module):
    def __init__(self, target):
        super().__init__()
        self.target = target.detach()

    def forward(self, x):
        self.loss = nn.functional.mse_loss(x, self.target)
        return x   

: 

In [ ]:
run_style_transfer(
    content_path="C:/Users/ADMIN/Downloads/pou/content.png",
    style_path="C:/Users/ADMIN/Downloads/pou/Screenshot 2026-03-11 214027.png",
    steps=400
)


Step 50 | Style: 0.0004 | Content: 7.9443
Step 100 | Style: 0.0001 | Content: 9.3942
Step 150 | Style: 0.0000 | Content: 9.2128
Step 200 | Style: 0.0000 | Content: 9.0485
Step 250 | Style: 0.0000 | Content: 8.9378
Step 300 | Style: 0.0000 | Content: 8.7119
Step 350 | Style: 0.0000 | Content: 8.2345
Step 400 | Style: 0.0000 | Content: 7.9279
